In [1]:
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *


In [2]:
ativos = ['titulos','hipotecas','emprestimos_veiculo','emprestimos_pessoais']

valor_total = 650000

retorno ={
    ativos[0]: 1.10,
    ativos[1]: 1.085, 
    ativos[2]: 1.095,
    ativos[3]: 1.125
}




In [8]:
model  = pyo.ConcreteModel()

model.ativos = pyo.Set(initialize=ativos)   
model.x = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)

def obj(model):
    return sum(model.x[a]*retorno[a] for a in model.ativos)
model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

def restricao_valor_total(model):
    return sum(model.x[a] for a in model.ativos) <= valor_total
model.restricao_valor_total = pyo.Constraint(rule=restricao_valor_total)

def restricao_emp_pessoal(model):
    return model.x['emprestimos_pessoais'] <= 0.25*valor_total
model.restricao_emp_pessoal = pyo.Constraint(rule=restricao_emp_pessoal)

def restricao_hipotecas(model):
    return model.x['hipotecas'] >= model.x['emprestimos_pessoais']
model.restricao_hipotecas = pyo.Constraint(rule=restricao_hipotecas)

def restricao_titulos(model):
    return model.x['titulos'] >= model.x['emprestimos_pessoais']
model.restricao_titulos = pyo.Constraint(rule=restricao_titulos)

In [9]:
model.pprint()

1 Set Declarations
    ativos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'titulos', 'hipotecas', 'emprestimos_veiculo', 'emprestimos_pessoais'}

1 Var Declarations
    x : Size=4, Index=ativos
        Key                  : Lower : Value : Upper : Fixed : Stale : Domain
        emprestimos_pessoais :     0 :  None :  None : False :  True : NonNegativeReals
         emprestimos_veiculo :     0 :  None :  None : False :  True : NonNegativeReals
                   hipotecas :     0 :  None :  None : False :  True : NonNegativeReals
                     titulos :     0 :  None :  None : False :  True : NonNegativeReals

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 1.1*x[titulos] + 1.085*x[hipotecas] + 1.095*x[emprestimos_veiculo] + 1.125*x[emprestimos_pessoais]

4 Constraint Declarations
    restricao_emp_

In [10]:
opt  = SolverFactory('cplex')
res=opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmplu8lw_14.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmp10o8mn40.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmp10o8mn40.pyomo.lp
Objective sense      : Maximize
Variables            :       4
Objective nonzeros   :       4
Linear constraints   :       4  [Less: 4]
  Nonzeros           :       9
  RHS nonzeros       :       2

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 1.085000         Max   : 1.125000    

In [ ]:
for m in model.x:
    print(f'Ativo: {m} = {model.x[m].value}')

print(f"Objetivo : {model.obj()}")

Ativo: titulos = 325000.0
Ativo: hipotecas = 162500.0
Ativo: emprestimos_veiculo = 0.0
Ativo: emprestimos_pessoais = 162500.0
Objetivo: 716625.0
